# Data Ingestion, Cleaning & Preprocessing with Pandas

This notebook cleans a messy retail sales dataset containing more than 10,000 records.

In [ ]:
import pandas as pd
import numpy as np
raw=pd.read_csv('raw_retail_sales_messy.csv')
print('Raw dataset shape:',raw.shape)
raw.head()

In [ ]:
print('Rows:',len(raw)); print('Columns:',len(raw.columns)); print('\nData types before cleaning:\n',raw.dtypes); print('\nMissing values before cleaning:\n',raw.isna().sum()); print('\nDuplicate rows:',raw.duplicated().sum())

In [ ]:
raw.describe(include='all').T

In [ ]:
df=raw.drop_duplicates().copy()
print('Rows before:',len(raw)); print('Rows after:',len(df)); print('Duplicates removed:',len(raw)-len(df))

In [ ]:
df['Order_Date']=pd.to_datetime(df['Order_Date'],dayfirst=True,errors='coerce')
df['Customer_ID']=df['Customer_ID'].fillna('UNKNOWN')
df['Product']=df['Product'].fillna(df['Product'].mode()[0])
df['Category']=df['Category'].fillna('Unknown')
df['Region']=df['Region'].astype('string').str.strip().str.title().fillna('Unknown')
df['Payment_Method']=df['Payment_Method'].fillna(df['Payment_Method'].mode()[0])
print('Categorical fields cleaned.')

In [ ]:
df['Quantity']=pd.to_numeric(df['Quantity'].astype('string').str.extract(r'(\d+(?:\.\d+)?)')[0],errors='coerce')
df['Quantity']=df['Quantity'].fillna(df['Quantity'].median()).round().clip(lower=1)
df['Unit_Price']=pd.to_numeric(df['Unit_Price'].astype('string').str.replace(r'[₹,\s]','',regex=True),errors='coerce').fillna(df['Unit_Price'].median())
for c in ['Sales','Cost']:
    df[c]=pd.to_numeric(df[c],errors='coerce').fillna(df[c].median())
df['Discount']=pd.to_numeric(df['Discount'],errors='coerce').fillna(0).clip(0,.5)
print(df.dtypes)

In [ ]:
df['Sales']=df['Quantity']*df['Unit_Price']*(1-df['Discount']); df['Cost']=df['Cost'].clip(lower=0); df['Profit']=df['Sales']-df['Cost']
for c in ['Sales','Profit']:
    q1,q3=df[c].quantile([.25,.75]); iqr=q3-q1; low,high=q1-1.5*iqr,q3+1.5*iqr; df[c]=df[c].clip(low,high)
print('Outliers treated using the IQR method.')

In [ ]:
df['Profit_Margin']=np.where(df['Sales']!=0,df['Profit']/df['Sales']*100,0)
df['Month']=df['Order_Date'].dt.month; df['Month_Name']=df['Order_Date'].dt.month_name(); df['Year']=df['Order_Date'].dt.year; df['Quarter']='Q'+df['Order_Date'].dt.quarter.astype(str)
df[['Order_Date','Month','Month_Name','Year','Quarter','Profit','Profit_Margin']].head()

In [ ]:
print('Clean dataset shape:',df.shape); print('\nMissing values after cleaning:\n',df.isna().sum()); print('\nDuplicate rows after cleaning:',df.duplicated().sum()); print('\nFinal data types:\n',df.dtypes)

In [ ]:
df.describe().T

In [ ]:
df=df.sort_values('Order_Date').reset_index(drop=True); df.to_csv('clean_dataset.csv',index=False); print('Saved clean_dataset.csv'); print('Final rows:',len(df)); print('Final columns:',len(df.columns))

## Conclusion

The raw retail data was cleaned by removing duplicates, handling missing values, standardizing data types and text, treating outliers with IQR, and creating month, year, quarter, profit, and profit-margin features.